## Парсинг данных с сайта на Python

### Определения

Парсинг (англ. parsing — разбор) — это процесс автоматического анализа веб-сайтов для сбора структурированной информации. Еще парсинг часто называют веб-скрапингом.

Парсинг — это процесс сбора данных с последующей их обработкой и анализом.

Программа, которая занимается парсингом, называется парсером.

### Условие задачи

С сайта ( https://habr.com/ru/search/ ) необходимо построить исходный набор данных (.csv или .xml). Набор данных должен включать __название, описание, рейтинг и сферу деятельности компаний, дату публикации, а также текст статей из Интернет-ресурсов__. Подготовленный набор данных должен содержать сведения о всех номинантах конкурса. Разработанный парсер должен извлекать гиперссылки из начальной страницы с последующим обходом всех страниц по полученным ссылкам и извлечением их содержимого. Можно дополнить набор какими-либо другими данными, если они могут быть полезны для дальнейшего исследования.


### Этапы парсинга

Работа состоит из нескольких основных этапов:

1. Поиск данных — получение HTML-кода страниц сайта Habr и поиск ссылок на статьи.
2. Получение информации — переход по найденным гиперссылкам и извлечение необходимых данных.
3. Обработка данных — очистка текста, обработка пропусков и исключение повторных ссылок.
4. Сохранение данных — формирование таблицы и сохранение результатов в CSV-файл.
5. Дополнительная обработка — сохранение части полученных данных в PDF и последующее извлечение информации из PDF.
6. Проверка качества — проверка полученных данных и формирование тестовых записей с пропущенными значениями.

### Подключение библиотек

In [1]:
!pip install beautifulsoup4 requests pandas reportlab PyPDF2

In [2]:
from bs4 import BeautifulSoup as bs

Beautiful Soup - это библиотека Python для извлечения данных из HTML и XML файлов. 

In [3]:
import requests

Библиотека requests является стандартным инструментом для составления HTTP-запросов в Python.

In [4]:
import pandas as pd
import time
import re
import os

### Получение информаций

In [5]:
url = 'https://habr.com/ru/all/'
page = requests.get(url)

Метод __requests.get(url)__ из библиотеки requests в Python выполняет HTTP-запрос типа GET по указанному URL. Этот запрос используется для получения данных с веб-страницы или API, в нашем случае из страницы habr.

In [6]:
page.status_code

200

Если вызвать __page.status_code__, то получим статус состояния HTTP. например, 200 — успешно, 404 — страница не найдена, 500 — ошибка сервера 


In [7]:
soup = bs(page.text, 'html.parser')

__bs__ — это сокращение от BeautifulSoup, основного класса библиотеки Beautiful Soup.

__bs(page.text, 'html.parser')__ создаёт объект BeautifulSoup, который парсит HTML-код из page.text с использованием указанного парсера.

__'html.parser'__ — это встроенный парсер Python, который не требует установки дополнительных библиотек.а.

In [8]:
page.text

'<!DOCTYPE html>\n<html lang="ru">\n\n  <head>\n    <title>Все статьи подряд &#x2F; Хабр</title>\n<link rel="image_src" href="/img/habr_ru.png" data-hid="2a79c45">\n<link href="https://habr.com/ru/articles/" rel="canonical" data-hid="e3fa780">\n<link href="https://habr.com/ru/articles/" hreflang="ru" rel="alternate" data-hid="7d51b8a">\n<link href="https://habr.com/en/articles/" hreflang="en" rel="alternate" data-hid="7d51b8a">\n<meta itemprop="image" content="/img/habr_ru.png">\n<meta property="og:image" content="/img/habr_ru.png">\n<meta property="og:image:width" content="1200">\n<meta property="og:image:height" content="630">\n<meta property="aiturec:image" content="/img/habr_ru.png">\n<meta name="twitter:image" content="/img/habr_ru.png">\n<meta property="vk:image" content="/img/habr_ru.png?format=vk">\n<meta property="fb:app_id" content="444736788986613">\n<meta property="fb:pages" content="472597926099084">\n<meta name="twitter:card" content="summary_large_image">\n<meta name="tw

In [9]:
soup

<!DOCTYPE html>

<html lang="ru">
<head>
<title>Все статьи подряд / Хабр</title>
<link data-hid="2a79c45" href="/img/habr_ru.png" rel="image_src"/>
<link data-hid="e3fa780" href="https://habr.com/ru/articles/" rel="canonical"/>
<link data-hid="7d51b8a" href="https://habr.com/ru/articles/" hreflang="ru" rel="alternate"/>
<link data-hid="7d51b8a" href="https://habr.com/en/articles/" hreflang="en" rel="alternate"/>
<meta content="/img/habr_ru.png" itemprop="image"/>
<meta content="/img/habr_ru.png" property="og:image"/>
<meta content="1200" property="og:image:width"/>
<meta content="630" property="og:image:height"/>
<meta content="/img/habr_ru.png" property="aiturec:image"/>
<meta content="/img/habr_ru.png" name="twitter:image"/>
<meta content="/img/habr_ru.png?format=vk" property="vk:image"/>
<meta content="444736788986613" property="fb:app_id"/>
<meta content="472597926099084" property="fb:pages"/>
<meta content="summary_large_image" name="twitter:card"/>
<meta content="@habr_com" name=

### Создание структуры данных

Создадим словарь, в который будем записывать данные по заданию: название статьи, описание, рейтинг и сферу деятельности компаний, дату публикации, а также текст статьи из Интернет-ресурса

In [11]:
result_list = {
    'title': [],
    'namecompany': [],
    'description': [],
    'rating': [],
    'field': [],
    'date': [],
    'textpub': []
}

### Алгоритм

Суть алгоритма заключается в переборе страниц, и переходе на "вложенные" страницы, то есть у нас есть основная страница https://habr.com/ru/all/, мы перебираем несколько стараниц, на каждой странице есть статьи, записываем их в список, чтобы перейти по ним используем  _-i.a.get('href')-_  то есть берём значение из href этого заголовка. Далее находим классы элементов которые нам нужны, и записываем их в результат.

In [12]:
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 '
                  '(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
}

def get_page(url, attempts=3, timeout=20):
    for attempt in range(attempts):
        try:
            response = requests.get(url, headers=headers, timeout=timeout)
            response.raise_for_status()
            return response
        except requests.RequestException:
            if attempt < attempts - 1:
                time.sleep(3)
            else:
                print(f"Не удалось загрузить страницу: {url}")
    return None


articles_collected = 0
seen_urls = set()

# Обрабатываем первые 5 страниц Habr
for pagenum in range(1, 6):

    if articles_collected >= 100:
        break

    list_url = f'https://habr.com/ru/all/page{pagenum}/'
    page = get_page(list_url)

    if page is None:
        continue

    soup = bs(page.text, 'html.parser')

    # Получаем статьи с текущей страницы
    titles = soup.find_all('h2', class_='tm-title tm-title_h2')

    for title in titles:

        if articles_collected >= 100:
            break

        if title.a is None:
            continue

        # Получаем ссылку на статью
        article_url = 'https://habr.com' + title.a.get('href')

        if article_url in seen_urls:
            continue

        seen_urls.add(article_url)
        time.sleep(1)

        # Загружаем страницу статьи
        article_page = get_page(article_url)

        if article_page is None:
            continue

        inner_soup = bs(article_page.text, 'html.parser')

        # Информация о компании
        name_company = inner_soup.find(
            'a', class_='tm-company-snippet__title'
        )

        desc_company = inner_soup.find(
            'div', class_='tm-company-snippet__description'
        )

        # Название статьи
        result_list['title'].append(title.text.strip())

        # Название компании
        result_list['namecompany'].append(
            name_company.text.strip() if name_company else ''
        )

        # Описание компании
        result_list['description'].append(
            desc_company.text.strip() if desc_company else ''
        )

        # Дата публикации
        datepub = inner_soup.find(
            'span', class_='tm-article-datetime-published'
        )

        if datepub and datepub.time:
            date_value = datepub.time.get('datetime', '')[:10]
        else:
            date_value = ''

        result_list['date'].append(date_value)

        # Текст статьи
        textpub = ''

        article_body = inner_soup.find(
            'div',
            class_='article-formatted-body article-formatted-body_version-2'
        )

        if article_body is None:
            article_body = inner_soup.find(
                'div',
                class_='article-formatted-body article-formatted-body_version-1'
            )

        if article_body:
            textpub = article_body.get_text(separator=' ', strip=True)

        result_list['textpub'].append(textpub)

        # Рейтинг и сфера деятельности компании
        rating_value = '0'
        field_value = ''

        if name_company is not None:
            company_href = name_company.get('href')

            if company_href:
                company_url = 'https://habr.com' + company_href
                time.sleep(1)

                company_page = get_page(company_url)

                if company_page is not None:
                    company_soup = bs(company_page.text, 'html.parser')

                    # Рейтинг компании
                    rating = company_soup.find(
                        'span',
                        class_='tm-votes-lever__score-counter '
                              'tm-votes-lever__score-counter_rating '
                              'tm-votes-lever__score-counter'
                    )

                    if rating:
                        rating_value = rating.text.strip()

                    # Сфера деятельности
                    fields = company_soup.find_all(
                        'a',
                        class_='tm-company-profile__categories-text'
                    )

                    field_value = ', '.join(
                        field.text.strip() for field in fields
                    )

        result_list['rating'].append(rating_value)
        result_list['field'].append(field_value)

        articles_collected += 1

print(f"\nСобрано статей: {articles_collected}")


Собрано статей: 99


### Проверка полученных данных

In [13]:
result_list

{'title': ['«Сейчас сроки получения патента были просто реактивными». Интервью с CEO Интеллитеха об участии в программе МИКа',
  'Где на самом деле живут знания компании: 8 видов корпоративного знания',
  'Хорошие и плохие русские в видеоиграх: действительно ли нас всегда делают злодеями?',
  'Фабрика кода с выключенным светом: почему Хорти зовет обратно читать код, и что у меня с этим сходится',
  'DotNext 2026',
  '65 бесплатных уроков октября: от LLM и Kubernetes до микросервисов, Kafka и безопасности',
  'Про магнитопроводы/сердечники катушек (в том числе, полностью самодельные)',
  'Периодическая таблица аббревиатур корпоративного ландшафта 2: ресурсы, аналитика и персонал',
  'За пределами EXPLAIN: как увидеть выполнение запроса вживую в распределённой СУБД',
  'Кем работать в\xa0IT в 2026: backend‑разработчик',
  'Кабельная проходка в nanoCAD BIM ОПС 25.1: от 3D-модели до спецификации',
  'Ну ИИ дела: как мы в Solar WAF хакеров под прикрытием DeepSeek и ChatGPT нашли',
  'Теплиц

In [14]:
print("Количество нулевых значений в: ")
for i in result_list:
    print( i + " - " + str(result_list[i].count(None)))

Количество нулевых значений в: 
title - 0
namecompany - 0
description - 0
rating - 0
field - 0
date - 0
textpub - 0


### Сохранение данных

In [15]:
df = pd.DataFrame(data=result_list)
df.to_csv('habr_20articles.csv', index=False, encoding='utf-8-sig')
print("Данные сохранены в habr_20articles.csv")

Данные сохранены в habr_20articles.csv


In [16]:
df.head(15)

,title,namecompany,description,rating,field,date,textpub
0,«Сейчас сроки получения патента были просто ре...,,,0,,2026-10-01,
1,Где на самом деле живут знания компании: 8 вид...,,,0,,2026-10-01,
2,Хорошие и плохие русские в видеоиграх: действи...,,,0,,2026-10-01,
3,Фабрика кода с выключенным светом: почему Хорт...,,,0,,2026-10-01,
4,DotNext 2026,,,0,,2026-10-01,
5,65 бесплатных уроков октября: от LLM и Kuberne...,,,0,,2026-10-01,
6,Про магнитопроводы/сердечники катушек (в том ч...,,,0,,2026-10-01,
7,Периодическая таблица аббревиатур корпоративно...,,,0,,2026-10-01,
8,За пределами EXPLAIN: как увидеть выполнение з...,,,0,,2026-10-01,
9,Кем работать в IT в 2026: backend‑разработчик,,,0,,2026-10-01,


In [17]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99 entries, 0 to 98
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   title        99 non-null     object
 1   namecompany  99 non-null     object
 2   description  99 non-null     object
 3   rating       99 non-null     object
 4   field        99 non-null     object
 5   date         99 non-null     object
 6   textpub      99 non-null     object
dtypes: object(7)
memory usage: 5.5+ KB


### Дополнительная обработка данных

В качестве дополнительной обработки часть собранных данных сохраняется в формате PDF.

Это позволяет дополнительно проверить работу с полученным набором данных и выполнить обратное извлечение структурированной информации из PDF-файла.

In [22]:
from reportlab.lib.pagesizes import A4
from reportlab.pdfgen import canvas
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont
from reportlab.lib.utils import simpleSplit

font_path = 'C:/Windows/Fonts/arial.ttf'

pdfmetrics.registerFont(TTFont('Arial', font_path))


def save_to_pdf(articles, filename='habr_articles.pdf'):
    c = canvas.Canvas(filename, pagesize=A4)

    width, height = A4

    for idx, article in enumerate(articles, 1):
        y = height - 50

        c.setFont('Arial', 12)

        title = f"{idx}. {article['title']}"
        title_lines = simpleSplit(title, 'Arial', 12, width - 100)

        for line in title_lines:
            c.drawString(50, y, line)
            y -= 18

        c.setFont('Arial', 10)

        information = [
            f"Компания: {article['namecompany']}",
            f"Рейтинг: {article['rating']}",
            f"Сфера: {article['field']}",
            f"Дата: {article['date']}",
            f"Описание: {article['description']}"
        ]

        for text in information:
            lines = simpleSplit(text, 'Arial', 10, width - 100)

            for line in lines:
                c.drawString(50, y, line)
                y -= 15

            y -= 5

            if y < 80:
                c.showPage()
                y = height - 50
                c.setFont('Arial', 10)

        c.showPage()

    c.save()

    print(f"PDF сохранён: {filename}")

In [23]:
articles_for_pdf = df.to_dict(orient='records')
save_to_pdf(articles_for_pdf, 'habr_articles.pdf')

PDF сохранён: habr_articles.pdf


### Извлечение данных из PDF

In [24]:
import PyPDF2
import re


def parse_pdf(
    filename='habr_articles.pdf'
):

    pdf_data = []

    with open(
        filename,
        'rb'
    ) as file:

        reader = PyPDF2.PdfReader(file)

        for page in reader.pages:

            text = page.extract_text()

            if not text:
                continue

            title_match = re.search(
                r'\d+\.\s*(.+)',
                text
            )

            company_match = re.search(
                r'Компания:\s*(.+)',
                text
            )

            rating_match = re.search(
                r'Рейтинг:\s*(.+)',
                text
            )

            field_match = re.search(
                r'Сфера:\s*(.+)',
                text
            )

            date_match = re.search(
                r'Дата:\s*(\d{4}-\d{2}-\d{2})',
                text
            )

            description_match = re.search(
                r'Описание:\s*(.+)',
                text
            )

            pdf_data.append({
                'title':
                    title_match.group(1).strip()
                    if title_match else '',

                'namecompany':
                    company_match.group(1).strip()
                    if company_match else '',

                'description':
                    description_match.group(1).strip()
                    if description_match else '',

                'rating':
                    rating_match.group(1).strip()
                    if rating_match else '',

                'field':
                    field_match.group(1).strip()
                    if field_match else '',

                'date':
                    date_match.group(1).strip()
                    if date_match else ''
            })

    return pdf_data


pdf_articles = parse_pdf(
    'habr_articles.pdf'
)

print(
    f"Спарсено из PDF: {len(pdf_articles)} статей"
)

Спарсено из PDF: 99 статей


### Проверка качества данных

Для проверки качества набора данных создаются дополнительные тестовые записи с пропущенными значениями.

Это позволяет проверить, как набор данных будет выглядеть при наличии неполной информации.

In [25]:
corrupted_data = [
    {
        'title': 'Статья без компании',
        'namecompany': '',
        'description': '',
        'rating': '0',
        'field': '',
        'date': '2026-03-09',
        'textpub': ''
    },
    {
        'title': '',
        'namecompany': 'Неизвестная компания',
        'description': 'Описание есть',
        'rating': '',
        'field': 'IT',
        'date': '',
        'textpub': ''
    }
]

### Формирование финального датасета

In [26]:
clean_data = pdf_articles[:10]

final_dataset = (
    clean_data +
    corrupted_data
)

df_final = pd.DataFrame(
    final_dataset
)

df_final.to_csv(
    'final_dataset.csv',
    index=False,
    encoding='utf-8-sig'
)

print(
    f"Финальный датасет: "
    f"{len(df_final)} записей"
)

print(
    f"Чистых: {len(clean_data)}"
)

print(
    f"С тестовыми ошибками: "
    f"{len(corrupted_data)}"
)

display(df_final)

Финальный датасет: 12 записей
Чистых: 10
С тестовыми ошибками: 2


,title,namecompany,description,rating,field,date,textpub
0,«Сейчас сроки получения патента были просто ре...,Рейтинг: 0,,0,Дата: 2026-10-01,2026-10-01,NaN
1,Где на самом деле живут знания компании: 8 вид...,Рейтинг: 0,,0,Дата: 2026-10-01,2026-10-01,NaN
2,Хорошие и плохие русские в видеоиграх: действи...,Рейтинг: 0,,0,Дата: 2026-10-01,2026-10-01,NaN
3,Фабрика кода с выключенным светом: почему Хорт...,Рейтинг: 0,,0,Дата: 2026-10-01,2026-10-01,NaN
4,DotNext 2026,Рейтинг: 0,,0,Дата: 2026-10-01,2026-10-01,NaN
5,65 бесплатных уроков октября: от LLM и Kuberne...,Рейтинг: 0,,0,Дата: 2026-10-01,2026-10-01,NaN
6,Про магнитопроводы/сердечники катушек (в том ч...,Рейтинг: 0,,0,Дата: 2026-10-01,2026-10-01,NaN
7,Периодическая таблица аббревиатур корпоративно...,Рейтинг: 0,,0,Дата: 2026-10-01,2026-10-01,NaN
8,За пределами EXPLAIN: как увидеть выполнение з...,Рейтинг: 0,,0,Дата: 2026-10-01,2026-10-01,NaN
9,Кем работать в IT в 2026: backend разработчик,Рейтинг: 0,,0,Дата: 2026-10-01,2026-10-01,NaN
